# Conjuntos de Pareto no domínio das variáveis de decisão

Este notebook representa, em uma matriz 3×3, os conjuntos de Pareto verdadeiros dos nove cenários no espaço tridimensional das variáveis de decisão $(x_1,x_2,x_3)$. Todos os painéis usam os mesmos limites cúbicos, projeção ortográfica e orientação isométrica.

Cada ponto é colorido pela média das respostas normalizadas do próprio cenário. Cada função é normalizada pelo ideal e pelo nadir verdadeiros antes do cálculo da média. Assim, azul representa valores médios próximos de 0 e vermelho valores próximos de 1. As estrelas vermelhas indicam os ótimos individuais verdadeiros, definidos pelas âncoras das funções.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
REFERENCE_DIR = ROOT / 'data' / 'reference_fronts'
SCENARIO_DIR = ROOT / 'data' / 'generated'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'true_pareto'
OUT_DIR.mkdir(parents=True, exist_ok=True)

M_VALUES = (4, 6, 12)
CORRELATION_LEVELS = ('low', 'medium', 'high')
DISPLAY_POINTS = 5000
N_STRATA = 40
PLOT_SEED = 20260825
CMAP = LinearSegmentedColormap.from_list(
    'BlueRedNoWhite',
    ['#2166AC', '#67A9CF', '#F4A582', '#B2182B'],
    N=256,
)
OUTPUT_STEM = 'nove_cenarios_pareto_dominio_x_isometria_media_normalizada'
FIGURE_WIDTH_CM = 16.0
FIGURE_HEIGHT_CM = 20.0
CM_TO_INCH = 1 / 2.54
ISOMETRIC_ELEVATION = float(np.degrees(np.arctan(1 / np.sqrt(2))))
ISOMETRIC_AZIMUTH = -45.0

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.0,
    'axes.titlesize': 10.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 6.5,
    'ytick.labelsize': 6.5,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('Raiz:', ROOT)
print('Saída:', OUT_DIR)

In [ ]:
def scenario_seed(scenario):
    return PLOT_SEED + sum((index + 1) * ord(char) for index, char in enumerate(scenario))

def load_scenario(scenario, expected_m):
    reference_path = REFERENCE_DIR / f'{scenario}_pareto_reference.npz'
    scenario_path = SCENARIO_DIR / f'{scenario}_scenario.npz'
    if not reference_path.exists() or not scenario_path.exists():
        raise FileNotFoundError(f'Artefatos ausentes para {scenario}.')
    with np.load(reference_path, allow_pickle=False) as reference:
        X = np.asarray(reference['X'], dtype=float)
        F = np.asarray(reference['F'], dtype=float)
        ideal = np.asarray(reference['ideal_true'], dtype=float)
        nadir = np.asarray(reference['nadir_true'], dtype=float)
    with np.load(scenario_path, allow_pickle=False) as scenario_data:
        anchors = np.asarray(scenario_data['anchors'], dtype=float)
    assert X.shape == (100_000, 3) and F.shape == (100_000, expected_m)
    assert anchors.shape == (expected_m, 3)
    assert ideal.shape == nadir.shape == (expected_m,)
    assert np.isfinite(X).all() and np.isfinite(F).all() and np.isfinite(anchors).all()
    span = nadir - ideal
    if np.any(span <= 0):
        raise ValueError(f'{scenario}: intervalo de normalização inválido.')
    normalized_F = np.clip((F - ideal) / span, 0.0, 1.0)
    mean_normalized = normalized_F.mean(axis=1)
    return X, mean_normalized, anchors, reference_path, scenario_path

def stratified_display_sample(X, score, scenario, n_points=DISPLAY_POINTS, n_strata=N_STRATA):
    if len(X) <= n_points:
        return X.copy(), score.copy()
    rng = np.random.default_rng(scenario_seed(scenario))
    edges = np.linspace(0.0, 1.0, n_strata + 1)
    strata = np.clip(np.digitize(score, edges[1:-1], right=False), 0, n_strata - 1)
    quota = n_points // n_strata
    chosen = []
    for stratum in range(n_strata):
        candidates = np.flatnonzero(strata == stratum)
        if len(candidates):
            take = min(quota, len(candidates))
            chosen.extend(rng.choice(candidates, size=take, replace=False).tolist())
    chosen = set(chosen)
    chosen.update((int(np.argmin(score)), int(np.argmax(score))))
    remaining = n_points - len(chosen)
    if remaining > 0:
        used = np.fromiter(chosen, dtype=int)
        pool = np.setdiff1d(np.arange(len(X)), used, assume_unique=False)
        chosen.update(rng.choice(pool, size=remaining, replace=False).tolist())
    indices = np.array(sorted(chosen), dtype=int)
    if len(indices) > n_points:
        protected = {int(np.argmin(score)), int(np.argmax(score))}
        removable = np.array(sorted(set(indices) - protected), dtype=int)
        keep = rng.choice(removable, size=n_points - len(protected), replace=False)
        indices = np.array(sorted(protected | set(keep.tolist())), dtype=int)
    assert len(indices) == n_points
    return X[indices], score[indices]

def shared_cube_limits(anchors_by_scenario, padding=0.06):
    all_anchors = np.vstack(list(anchors_by_scenario.values()))
    minima = all_anchors.min(axis=0)
    maxima = all_anchors.max(axis=0)
    centers = 0.5 * (minima + maxima)
    half_width = 0.5 * np.max(maxima - minima) * (1 + 2 * padding)
    return tuple((center - half_width, center + half_width) for center in centers)

def style_isometric_axis(ax, scenario, cube_limits):
    ax.set_xlim(*cube_limits[0])
    ax.set_ylim(*cube_limits[1])
    ax.set_zlim(*cube_limits[2])
    ax.set_box_aspect((1, 1, 1))
    ax.set_proj_type('ortho')
    ax.view_init(elev=ISOMETRIC_ELEVATION, azim=ISOMETRIC_AZIMUTH)
    ax.set_title(scenario, pad=-3)
    ax.set_xlabel(r'$x_1$', labelpad=-4)
    ax.set_ylabel(r'$x_2$', labelpad=-4)
    ax.set_zlabel('')
    ax.text2D(0.94, 0.50, r'$x_3$', transform=ax.transAxes, rotation=90, ha='center', va='center')
    for setter, limits in zip((ax.set_xticks, ax.set_yticks, ax.set_zticks), cube_limits):
        setter(np.linspace(limits[0], limits[1], 3))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
        axis.pane.set_edgecolor((0.65, 0.65, 0.65, 1.0))
        axis._axinfo['grid']['color'] = (0.82, 0.82, 0.82, 0.75)
        axis._axinfo['grid']['linewidth'] = 0.45

In [ ]:
loaded = {}
anchors_by_scenario = {}
for m in M_VALUES:
    for correlation in CORRELATION_LEVELS:
        scenario = f'm{m}_{correlation}'
        loaded[scenario] = load_scenario(scenario, m)
        anchors_by_scenario[scenario] = loaded[scenario][2]

cube_limits = shared_cube_limits(anchors_by_scenario)
color_norm = Normalize(vmin=0.0, vmax=1.0)
fig, axes = plt.subplots(
    3, 3,
    figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH),
    subplot_kw={'projection': '3d'},
)
fig.subplots_adjust(left=0.035, right=0.985, top=0.975, bottom=0.135, wspace=0.015, hspace=0.16)
manifest_rows = []

for row, m in enumerate(M_VALUES):
    for column, correlation in enumerate(CORRELATION_LEVELS):
        scenario = f'm{m}_{correlation}'
        X, score, anchors, reference_path, scenario_path = loaded[scenario]
        shown_X, shown_score = stratified_display_sample(X, score, scenario)
        ax = axes[row, column]
        ax.scatter(
            shown_X[:, 0], shown_X[:, 1], shown_X[:, 2],
            c=shown_score, cmap=CMAP, norm=color_norm,
            s=2.0, alpha=0.34, linewidths=0, depthshade=False, rasterized=True,
        )
        ax.scatter(
            anchors[:, 0], anchors[:, 1], anchors[:, 2],
            marker='*', s=72, c='red', edgecolors='0.12', linewidths=0.55,
            depthshade=False, zorder=10,
        )
        style_isometric_axis(ax, scenario, cube_limits)
        manifest_rows.append({
            'scenario': scenario,
            'm': m,
            'correlation_level': correlation,
            'reference_points': len(X),
            'display_points': len(shown_X),
            'individual_optima': len(anchors),
            'color_variable': 'mean_of_all_normalized_objectives',
            'normalization': '(f_j - ideal_true_j) / (nadir_true_j - ideal_true_j)',
            'reference_source': reference_path.relative_to(ROOT).as_posix(),
            'anchor_source': scenario_path.relative_to(ROOT).as_posix(),
        })

scalar_mappable = plt.cm.ScalarMappable(norm=color_norm, cmap=CMAP)
scalar_mappable.set_array([])
grid_left = min(ax.get_position().x0 for ax in axes.flat)
grid_right = max(ax.get_position().x1 for ax in axes.flat)
colorbar_axis = fig.add_axes([grid_left, 0.065, grid_right - grid_left, 0.016])
colorbar = fig.colorbar(scalar_mappable, cax=colorbar_axis, orientation='horizontal')
colorbar.set_label('Média das funções normalizadas (0 = ótimo; 1 = pior)', labelpad=8)
colorbar.set_ticks(np.linspace(0, 1, 6))

png_path = OUT_DIR / f'{OUTPUT_STEM}.png'
pdf_path = OUT_DIR / f'{OUTPUT_STEM}.pdf'
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path, dpi=300)
plt.close(fig)

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / f'{OUTPUT_STEM}_manifest.csv'
manifest.to_csv(manifest_path, index=False)
metadata = {
    'layout': '3x3; rows m=4,6,12; columns low,medium,high',
    'domain': ['x1', 'x2', 'x3'],
    'projection': 'orthographic isometric',
    'view_elevation_degrees': ISOMETRIC_ELEVATION,
    'view_azimuth_degrees': ISOMETRIC_AZIMUTH,
    'shared_cube_limits': [list(pair) for pair in cube_limits],
    'color': 'mean of all normalized objectives; continuous blue-to-red scale without white, blue at 0 and red at 1',
    'individual_optima': 'red stars at scenario anchors',
    'display_sampling': f'deterministic and stratified by color variable, {DISPLAY_POINTS} points per panel',
    'seed': PLOT_SEED,
    'publication_size_cm': [FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM],
    'png': png_path.relative_to(ROOT).as_posix(),
    'pdf': pdf_path.relative_to(ROOT).as_posix(),
}
metadata_path = OUT_DIR / f'{OUTPUT_STEM}_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

for artifact in (png_path, pdf_path, manifest_path, metadata_path):
    assert artifact.exists() and artifact.stat().st_size > 0
assert len(manifest) == 9 and set(manifest['reference_points']) == {100_000}
print(manifest.to_string(index=False))
print('Limites cúbicos compartilhados:', cube_limits)
print('Arquivos gerados:')
for artifact in (png_path, pdf_path, manifest_path, metadata_path):
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura da figura

- Cada nuvem representa o conjunto de Pareto verdadeiro no domínio das variáveis de decisão.
- A mesma orientação isométrica e o mesmo cubo de limites são usados nos nove cenários, permitindo comparar posição, extensão e geometria sem distorção de escala entre painéis.
- A cor resume o desempenho multiobjetivo médio depois que cada função foi normalizada individualmente pelo ideal e pelo nadir verdadeiros.
- As estrelas vermelhas representam as âncoras e, portanto, os ótimos individuais verdadeiros das funções.
- A subamostragem altera somente a densidade visual; cada arquivo de referência original continua contendo 100.000 pontos.